# **Part 3 — Machine Learning: predict no-shows and choose the model honestly**

# Task 3.1 — Features, split and baseline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [ ]:
df = pd.read_csv("/content/appointments_clean.csv")

In [ ]:
df.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [ ]:
df.columns

Index(['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
       'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel',
       'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days',
       'no_show'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   990 non-null    object 
 1   patient_id       990 non-null    object 
 2   clinic_id        990 non-null    object 
 3   booked_on        990 non-null    object 
 4   appt_date        990 non-null    object 
 5   age              990 non-null    int64  
 6   gender           990 non-null    object 
 7   distance_km      990 non-null    float64
 8   fee_inr          990 non-null    int64  
 9   booking_channel  990 non-null    object 
 10  visit_type       990 non-null    object 
 11  reminder_sent    990 non-null    object 
 12  prior_no_shows   990 non-null    int64  
 13  status           990 non-null    object 
 14  lead_days        990 non-null    int64  
 15  no_show          990 non-null    int64  
dtypes: float64(1), int64(5), object(10)
memory usage: 123.9+ KB


In [ ]:
features = [
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows",
    "reminder_sent",
    "booking_channel",
    "visit_type"
]

X = df[features].copy()
y = df["no_show"]

In [ ]:
X["reminder_sent"] = X["reminder_sent"].map({"Yes":1, "No":0})

In [ ]:
X = pd.get_dummies(X, columns = ["booking_channel","visit_type"],dtype = int)


In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [ ]:
attended_in_test = int((y_test == 0).sum())
baseline_accuracy = attended_in_test / len(y_test)

In [ ]:
print("X has", X.shape[1],"columns")
print("train",len(X_train),"rows")
print("test",len(X_test),"rows")
print(f"{int(y_test.sum())} no-shows in the test set ({y.mean() * 100:.1f}% of the data overall)")
print(f"Baseline accuracy: {baseline_accuracy:.4f} ({attended_in_test} of {len(y_test)})")

X has 11 columns
train 792 rows
test 198 rows
49 no-shows in the test set (22.0% of the data overall)
Baseline accuracy: 0.7525 (149 of 198)


# Task 3.2 — Logistic regression

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()


In [ ]:
# StandardScaler fit on the training set
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
# 2) LogisticRegression
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter = 1000)

model.fit(X_train_s, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred = model.predict(X_test_s)

In [ ]:
y_pred

array([0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0])

In [ ]:
y_prob = model.predict_proba(X_test_s)[:, 1]

In [ ]:
y_test

,no_show
213,0
331,0
501,0
309,0
88,0
...,...
450,1
705,0
305,0
807,1


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


In [ ]:
accuracy_score(y_test, y_pred)

0.7878787878787878

In [ ]:
precision_score(y_test, y_pred)


0.6296296296296297

In [ ]:
recall_score(y_test, y_pred)

0.3469387755102041

In [ ]:
f1_score(y_test, y_pred)

0.4473684210526316

In [ ]:
roc_auc_score(y_test, y_prob)

np.float64(0.788933022873579)

In [ ]:
confusion_matrix(y_test, y_pred)

array([[139,  10],
       [ 32,  17]])

In [ ]:
# 1. Print coefficients in column order of X
coef_df = pd.DataFrame({
    "feature" : X.columns,
    "coefficient": model.coef_[0]
})

# 2. Odds ratios for reminder_sent (scaled vs. unscaled 0 -> 1)
idx = list(X.columns).index("reminder_sent")
coef_rem = model.coef_[0][idx]
scale_rem = scaler.scale_[idx]

scaled_odd_ratio = np.exp(coef_rem)
unscaled_odd_ratio = np.exp(coef_rem / scale_rem)

print(f"\nreminder_sent SD (scale): {scale_rem:.4f}")
print(f"Scaled np.exp(coef): {scaled_odd_ratio:.4f}")
print(f"Unscaled np.exp(coef / scale): {unscaled_odd_ratio:.4f}")



reminder_sent SD (scale): 0.4950
Scaled np.exp(coef): 0.5595
Unscaled np.exp(coef / scale): 0.3094


**Interpretation & Conclusion:**


---



* **Feature Effects:** `reminder_sent` has the most negative coefficient ($\approx -0.58$); `lead_days` and `prior_no_shows` are both clearly positive ($\approx +0.6$).
* **Odds Ratios & Why They Differ:** `np.exp` of the `reminder_sent` coefficient is $\approx 0.56$ (odds $\times 0.56$ per one standard deviation of the scaled feature, $\text{SD} \approx 0.49$), whereas for the real `No` $\rightarrow$ `Yes` change, `np.exp(coef / scale)` is $\approx 0.31$ because rescaling converts the effect from a 1-SD step to a full 1-unit change.
* **Conclusion:** In this fitted model, holding the other features constant, the odds of a no-show are about **69% lower** for appointments with a reminder ($\text{odds ratio} \approx 0.31$). Because the data are observational, this is an association rather than proof of cause (and we should not report 44% as the effect of sending a reminder).

Task 3.3 — Decision tree and overfitting

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
model_DT = DecisionTreeClassifier(random_state=42)
model_DT.fit(X_train_s, y_train)

DecisionTreeClassifier(random_state=42)

In [ ]:
y_pred_train = model_DT.predict(X_train_s)
y_pred_test = model_DT.predict(X_test_s)

In [ ]:
accuracy_score(y_train,y_pred_train)


1.0

In [ ]:
accuracy_score(y_test,y_pred_test)

0.6717171717171717

In [ ]:
model_DT = DecisionTreeClassifier(max_depth = 3,random_state=42)
model_DT.fit(X_train_s, y_train)

DecisionTreeClassifier(max_depth=3, random_state=42)

In [ ]:
y_pred_train = model_DT.predict(X_train_s)
y_pred_test = model_DT.predict(X_test_s)

In [ ]:
accuracy_score(y_train,y_pred_train)

0.821969696969697

In [ ]:
accuracy_score(y_test, y_pred_test)

0.7777777777777778

**Conclusion:** The large gap between training accuracy1.00 and test accuracy 0.67 shows that the unconstrained tree severely overfits by memorizing noise, whereas limiting max_depth=3 narrows the gap (0.82 vs. 0.78) and improves generalization on unseen appointments.

# Task 3.4 — Random forest and a comparison table

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model_RFC = RandomForestClassifier(n_estimators=100, random_state=42)           #n_estimator, default=100-  The number of trees in the forest.

model_RFC.fit(X_train_s, y_train)

RandomForestClassifier(random_state=42)

In [ ]:
y_predict_train = model_RFC.predict(X_train_s)
y_predict_test = model_RFC.predict(X_test_s)

In [ ]:
accuracy_score(y_train,y_predict_train)

1.0

In [ ]:
accuracy_score(y_test,y_predict_test)


0.7474747474747475

In [ ]:
recall_score(y_test, y_predict_test)

0.30612244897959184

In [ ]:
y_prob_rfc = model_RFC.predict_proba(X_test_s)[:, 1]
roc_auc_score(y_test, y_prob_rfc)

np.float64(0.7077112724284345)

In [ ]:
feature_importances = pd.Series(model_RFC.feature_importances_, index = X.columns)
top_3_features = feature_importances.sort_values(ascending = False).head(3)
top_3_features

,0
lead_days,0.219905
age,0.196788
distance_km,0.194739


In [ ]:
# 1. Logistic Regression
lr_pred = model.predict(X_test_s)
lr_prob = model.predict_proba(X_test_s)[:, 1]                                   #[:, 1]- Select column index 1 (No-Show probability) across all test rows; index 0 is Attended

# 2. Depth-3 Decision Tree
dt_pred = model_DT.predict(X_test_s)
dt_prob = model_DT.predict_proba(X_test_s)[:, 1]

# 3. Random Forest
rf_pred = model_RFC.predict(X_test_s)
rf_prob = model_RFC.predict_proba(X_test_s)[:, 1]

# 4. Three Models: Accuracy, Recall आणि ROC-AUC Table
comparison_df = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": round(accuracy_score(y_test, lr_pred), 4),
        "Recall": round(recall_score(y_test, lr_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, lr_prob), 4),
    },
    {
        "Model": "Decision Tree (max_depth=3)",
        "Accuracy": round(accuracy_score(y_test, dt_pred), 4),
        "Recall": round(recall_score(y_test, dt_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, dt_prob), 4),
    },
    {
        "Model": "Random Forest",
        "Accuracy": round(accuracy_score(y_test, rf_pred), 4),
        "Recall": round(recall_score(y_test, rf_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, rf_prob), 4),
    },
])

print(comparison_df.to_string(index=False))

                      Model  Accuracy  Recall  ROC-AUC
        Logistic Regression    0.7879  0.3469   0.7889
Decision Tree (max_depth=3)    0.7778  0.2245   0.6997
              Random Forest    0.7475  0.3061   0.7077


**Conclusion:**
Logistic regression has the highest test ROC-AUC of the three models

# Task 3.5 — Tune, choose the model and the threshold, then test once

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline


(a) Tune Random Forest with GridSearchCV

In [ ]:
# (a) Tune Random Forest hyperparameters (number of trees and tree depth) using 5-fold cross-validation(cv)
# Goal: Find the best settings that give the highest ROC-AUC score on the training data

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8],
}

# Run 5-fold cross-validated grid search optimizing ROC-AUC
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), cv=5,param_grid=param_grid, scoring="roc_auc",)

# Fit on the training set (X_train_s, y_train are already defined in your notebook)
rf_grid.fit(X_train_s, y_train)



GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
             param_grid={'max_depth': [3, 5, 8], 'n_estimators': [100, 200]},
             scoring='roc_auc')

In [ ]:
print("Best params:", rf_grid.best_params_)
print("Best CV ROC-AUC:", rf_grid.best_score_)

Best params: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.7668976597090449


(b) Choose the Final Model (Without Touching Test Data)

In [ ]:
# (b) Compare all 4 models on the training set using 5-fold cross-validated ROC-AUC
# Note: Use a Pipeline for Logistic Regression so scaling happens separately inside each fold (preventing data leakage)
# Result: Logistic Regression wins with the highest CV ROC-AUC (~0.7737), so it is selected as our final model

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Logistic Regression using the supplied pipeline on unscaled X_train
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
cv_auc = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()

# 2. Depth-3 Decision Tree (model_DT from cell [62])
dt_cv_auc = cross_val_score(model_DT, X_train, y_train, cv=5, scoring="roc_auc").mean()

# 3. Untuned Random Forest (model_RFC from cell [68])
rf_cv_auc = cross_val_score(model_RFC, X_train, y_train, cv=5, scoring="roc_auc").mean()

# 4. Tuned Random Forest best_score_ from GridSearchCV (cell [83])
tuned_rf_cv_auc = rf_grid.best_score_

# Print all 5-fold CV ROC-AUC scores on the training set
print(f"Logistic Regression (Pipeline) 5-Fold CV ROC-AUC: {cv_auc:.4f}")
print(f"Decision Tree (max_depth=3)    5-Fold CV ROC-AUC: {dt_cv_auc:.4f}")
print(f"Random Forest (n_estimators=100) 5-Fold CV ROC-AUC: {rf_cv_auc:.4f}")
print(f"Tuned Random Forest (GridSearchCV) 5-Fold CV ROC-AUC: {tuned_rf_cv_auc:.4f}")

Logistic Regression (Pipeline) 5-Fold CV ROC-AUC: 0.7737
Decision Tree (max_depth=3)    5-Fold CV ROC-AUC: 0.6784
Random Forest (n_estimators=100) 5-Fold CV ROC-AUC: 0.7466
Tuned Random Forest (GridSearchCV) 5-Fold CV ROC-AUC: 0.7669


(c) Choose the Best Probability Threshold

In [ ]:
# (c) Get out-of-fold predicted probabilities on the training set using the winning model (lr_pipe)
# Compare threshold 0.5 vs 0.35 on Precision, Recall, F1-score, and total flagged appointments
# Decision: Choose 0.35 because it gives a higher F1-score and catches more actual no-shows (higher Recall)
# Why Recall matters: Missing a no-show wastes an entire doctor slot, while making an extra reminder call takes only a minute

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv =5, method = "predict_proba")[:, 1]    # Compare default threshold (0.5) vs lower threshold (0.35) to catch more potential no-shows

for threshold in [0.5, 0.35]:
    preds = (oof_probs >= threshold).astype(int)
    p = precision_score(y_train, preds)
    r = recall_score(y_train, preds)
    f1 = f1_score(y_train, preds)
    flagged = preds.sum()

    print(f"Threshold: {threshold}")
    print(f"Precision: {p:.4f}, Recall: {r:.4f}, F1: {f1:.4f}, Flagged: {flagged}")



Threshold: 0.5
Precision: 0.5753, Recall: 0.2485, F1: 0.3471, Flagged: 73
Threshold: 0.35
Precision: 0.5130, Recall: 0.4675, F1: 0.4892, Flagged: 154


(d) Final One-Time Evaluation on the Test Set

In [ ]:
# (d) Train the winning Logistic Regression pipeline on the full training set (X_train, y_train)
# Evaluate it ONCE on the unseen test set (X_test) using our chosen threshold of 0.35
# Report final Test Precision, Recall, number of flagged appointments, and ROC-AUC

lr_pipe.fit(X_train, y_train)
test_probs = lr_pipe.predict_proba(X_test)[:,1]
chosen_threshold = 0.35
test_flags = (test_probs >= chosen_threshold).astype(int)

print("\nFinal Test Evalution(at 0.35):")
print(f"Precision: {precision_score(y_test, test_flags):.2f}")
print(f"Recall:    {recall_score(y_test, test_flags):.2f}")
print(f"Flagged:   {test_flags.sum()}")
print(f"ROC-AUC:   {roc_auc_score(y_test, test_probs):.2f}")



Final Test Evalution(at 0.35):
Precision: 0.57
Recall:    0.57
Flagged:   49
ROC-AUC:   0.79


(e) Save Test Predictions to CSV & Practical Takeaway

In [ ]:
# (e) Create a final results table for all 198 test appointments with appointment_id, actual no_show, risk_prob, and flag (1 if risk_prob >= 0.35)
# Export the table to 'test_risk_scores.csv' for front-desk follow-up calls
# Practical takeaway: In real life, a clinic sets this threshold by balancing how many evening calls the staff can make against the cost of an empty doctor slot

results_df = pd.DataFrame({
    "appointment_id" : df.loc[X_test.index, "appointment_id"],                  # Use X_test.index (row positions) to fetch the real 'appointment_id' text values from df
                                                                                # (needed because 'appointment_id' was removed from X before training the model)
    "no_show" : y_test,
    "risk_prob" : test_probs,
    "flag" : test_flags
})

# Save the final 198 test predictions to CSV
results_df.to_csv("/content/test_risk_scores.csv", index = False)

# Verify the saved output matches 198 rows and 49 flagged rows
print(f"test_risk_scores.csv has {len(results_df)} rows, and the number of rows with flag = 1 is {int(results_df['flag'].sum())} (matches reported: {int(test_flags.sum())}).")


test_risk_scores.csv has 198 rows, and the number of rows with flag = 1 is 49 (matches reported: 49).
